# 🏗️ Day 04: Tree Construction — Building Trees from Scratch

---

## The Key Insight for Construction Problems

To uniquely reconstruct a binary tree, you need **two** traversals (one must be inorder). Why?

- **Preorder** tells you the root (first element)
- **Inorder** tells you what's on the left vs right of that root

With both pieces of info, you can recursively split and build.

---

In [ ]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

from collections import deque

def print_tree(root, level=0, prefix="Root: "):
    if root is not None:
        print(" " * (level * 4) + prefix + str(root.val))
        if root.left or root.right:
            print_tree(root.left, level + 1, "L--- ")
            print_tree(root.right, level + 1, "R--- ")

def inorder(root):
    if not root: return []
    return inorder(root.left) + [root.val] + inorder(root.right)

## 🎯 Problem 1: Construct BT from Preorder & Inorder (LC #105)

```
Preorder: [3, 9, 20, 15, 7]   (root is always first: 3)
Inorder:  [9, 3, 15, 20, 7]   (everything left of 3 = left subtree: [9])
                                (everything right of 3 = right subtree: [15,20,7])
```

**Algorithm:**
1. First element of preorder = root
2. Find root in inorder → splits into left and right subtrees
3. Recurse on each half

**Optimization:** Use a hashmap for O(1) lookup of root position in inorder.

In [ ]:
def buildTree_pre_in(preorder, inorder):
    """
    Build tree from preorder + inorder.
    Time: O(n), Space: O(n)
    """
    # Map value → index in inorder for O(1) lookup
    inorder_map = {val: i for i, val in enumerate(inorder)}
    pre_idx = [0]  # Using list to allow mutation in nested function
    
    def build(in_left, in_right):
        if in_left > in_right:
            return None
        
        # Pick current root from preorder
        root_val = preorder[pre_idx[0]]
        pre_idx[0] += 1
        root = TreeNode(root_val)
        
        # Find root position in inorder
        mid = inorder_map[root_val]
        
        # Build left subtree first (matches preorder traversal order)
        root.left = build(in_left, mid - 1)
        root.right = build(mid + 1, in_right)
        
        return root
    
    return build(0, len(inorder) - 1)

tree = buildTree_pre_in([3, 9, 20, 15, 7], [9, 3, 15, 20, 7])
print_tree(tree)
print(f"Inorder check: {inorder(tree)}")  # [9, 3, 15, 20, 7]

---

## 🎯 Problem 2: Construct BT from Inorder & Postorder (LC #106)

Same idea, but root is the **last** element of postorder. And we build **right subtree first** (because postorder goes left→right→root, so reading backwards gives root→right→left).

In [ ]:
def buildTree_in_post(inorder, postorder):
    """
    Build tree from inorder + postorder.
    Time: O(n), Space: O(n)
    """
    inorder_map = {val: i for i, val in enumerate(inorder)}
    post_idx = [len(postorder) - 1]  # Start from end
    
    def build(in_left, in_right):
        if in_left > in_right:
            return None
        
        root_val = postorder[post_idx[0]]
        post_idx[0] -= 1
        root = TreeNode(root_val)
        
        mid = inorder_map[root_val]
        
        # Build RIGHT first! (postorder reversed = root→right→left)
        root.right = build(mid + 1, in_right)
        root.left = build(in_left, mid - 1)
        
        return root
    
    return build(0, len(inorder) - 1)

tree = buildTree_in_post([9, 3, 15, 20, 7], [9, 15, 7, 20, 3])
print_tree(tree)

---

## 🎯 Problem 3: Serialize and Deserialize Binary Tree (LC #297)

Convert a tree to a string and back. This is a hard problem but the approach is clean.

**Strategy:** Preorder traversal with "null" markers. The null markers tell us exactly where subtrees end, so we can reconstruct without needing inorder.

In [ ]:
class Codec:
    def serialize(self, root):
        """Preorder with null markers."""
        result = []
        def dfs(node):
            if not node:
                result.append("N")
                return
            result.append(str(node.val))
            dfs(node.left)
            dfs(node.right)
        dfs(root)
        return ",".join(result)
    
    def deserialize(self, data):
        """Rebuild from preorder + null markers."""
        values = iter(data.split(","))
        
        def dfs():
            val = next(values)
            if val == "N":
                return None
            node = TreeNode(int(val))
            node.left = dfs()
            node.right = dfs()
            return node
        
        return dfs()

codec = Codec()

# Build a tree, serialize, then deserialize
original = buildTree_pre_in([1, 2, 3, 4, 5], [3, 2, 4, 1, 5])
serialized = codec.serialize(original)
print(f"Serialized: {serialized}")

restored = codec.deserialize(serialized)
print(f"\nRestored tree:")
print_tree(restored)
print(f"Inorder matches: {inorder(original) == inorder(restored)}")  # True

**Why preorder + null markers works:** The null markers explicitly mark where each subtree ends, so every value knows exactly which subtree it belongs to. No ambiguity.

---

## 🗺️ Construction Pattern Map

```
╔══════════════════════════════════════════════════════════════════════╗
║  TREE CONSTRUCTION PATTERNS                                         ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. PRE + IN: Root from preorder[0], split inorder, left first     ║
║  2. POST + IN: Root from postorder[-1], split inorder, right first ║
║  3. SERIALIZE: Preorder + null markers → no inorder needed         ║
║                                                                      ║
║  KEY OPTIMIZATION: HashMap for O(1) inorder index lookup            ║
║  KEY INSIGHT: Inorder splits left/right. Pre/Post gives root.       ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Key Idea |
|---|---------|------------|----------|
| 105 | Construct BT from Pre + In | Medium | Root first, split by inorder |
| 106 | Construct BT from In + Post | Medium | Root last, build right first |
| 297 | Serialize/Deserialize BT | Hard | Preorder + null markers |
| 654 | Maximum Binary Tree | Medium | Max as root, recurse left/right |
| 108 | Sorted Array to BST | Easy | Mid = root, recurse halves |
| 1008 | Construct BST from Preorder | Medium | Bound-based construction |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 Need 2 traversals to rebuild (one must be inorder)             │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Pre+In: build left first. Post+In: build right first.         │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Serialize with null markers → single traversal enough          │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Always use hashmap for O(1) inorder index lookup               │
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Path Problems

Path sums, diameter, lowest common ancestor — the most popular interview problems.